# Limpieza de datos: Hábitos y tiempo de uso de IA en estudiantes

Este notebook toma el CSV exportado del formulario y genera `datos_limpios_ia.csv`.

**Pipeline**

0. Carga e inspección inicial
1. Estandarización de columnas
2. Limpieza de texto y categorías
3. Conversión de tipos (casteo)
4. Datos faltantes
5. Datos duplicados
6. Control de rangos
7. Formato de presentación para el dashboard
8. Exportación

Cada paso tiene una nota con **qué se decidió y por qué**. El orden importa y se justifica en cada sección.

In [11]:
import re

import numpy as np
import pandas as pd

# ---- Configuración ----
RUTA_ENTRADA = "Encuesta sobre Hábitos y Tiempo de Uso de Inteligencia Artificial  (Respuestas) - Respuestas de formulario 1.csv"
RUTA_SALIDA = "datos_limpios_ia.csv"

# Diccionario de renombrado (claves = pregunta original SIN espacios en los extremos)
RENOMBRAR = {
    "Marca temporal": "marca_temporal",
    "1. ¿Cuál es tu edad?": "edad",
    "2. ¿Qué herramienta de IA utilizas con mayor frecuencia?": "herramienta_ia",
    "3. ¿Cuál es la finalidad principal por la que utilizas esta herramienta?": "finalidad",
    "4. ¿Cómo calificarías tu nivel de experiencia utilizando herramientas de IA?": "nivel_experiencia",
    "5. ¿Cuántos días a la semana utilizas herramientas de IA?": "frecuencia_semanal",
    "6. En los días que utilizas IA, ¿cuánto tiempo aproximado (en minutos) le dedicas por día?": "tiempo_uso",
    "7. De todas las horas que dedicás a estudiar en la semana, ¿qué porcentaje de ese tiempo usás Inteligencia Artificial?": "porcentaje_estudio_ia",
}

COLS_NUMERICAS = ["edad", "frecuencia_semanal", "tiempo_uso", "porcentaje_estudio_ia"]
COLS_CATEGORICAS = ["herramienta_ia", "finalidad", "nivel_experiencia"]
COLS_CRITICAS = ["herramienta_ia", "tiempo_uso", "frecuencia_semanal"]

## 0. Carga e inspección inicial

Se carga el archivo **sin modificarlo** y se mira su forma, tipos y valores antes de decidir nada.
Los pasos siguientes se basan en lo que se observa acá.

In [12]:
df_raw = pd.read_csv(RUTA_ENTRADA)
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
df_raw.info()
df_raw.head()

Filas: 37 | Columnas: 8
<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 8 columns):
 #   Column                                                                                                                  Non-Null Count  Dtype
---  ------                                                                                                                  --------------  -----
 0   Marca temporal                                                                                                          37 non-null     str  
 1   1. ¿Cuál es tu edad?                                                                                                    37 non-null     int64
 2   2. ¿Qué herramienta de IA utilizas con mayor frecuencia?                                                                37 non-null     str  
 3   3. ¿Cuál es la finalidad principal por la que utilizas esta herramienta?                                                37 non-null     str  
 4 

,Marca temporal,1. ¿Cuál es tu edad?,2. ¿Qué herramienta de IA utilizas con mayor frecuencia?,3. ¿Cuál es la finalidad principal por la que utilizas esta herramienta?,4. ¿Cómo calificarías tu nivel de experiencia utilizando herramientas de IA?,5. ¿Cuántos días a la semana utilizas herramientas de IA?,"6. En los días que utilizas IA, ¿cuánto tiempo aproximado (en minutos) le dedicas por día?","7. De todas las horas que dedicás a estudiar en la semana, ¿qué porcentaje de ese tiempo usás Inteligencia Artificial?"
0,9/10/2026 20:39:47,27,Claude,"Resolver dudas o conceptos de estudio., Escrib...","Avanzado (Uso diario, integración en flujos de...",5,30,50
1,9/10/2026 20:40:15,20,Gemini,Entretenimiento.,Básico (Uso esporádico y consultas simples).,5,30,45
2,9/10/2026 20:45:11,20,Gemini,Resolver dudas o conceptos de estudio.,Básico (Uso esporádico y consultas simples).,2,20,50
3,9/10/2026 20:57:06,33,Gemini,"Resolver dudas o conceptos de estudio., Escrib...","Intermedio (Uso regular, prompts detallados).",5,60,50
4,9/10/2026 21:01:08,28,ChatGPT,"Resolver dudas o conceptos de estudio., Redacc...","Intermedio (Uso regular, prompts detallados).",3,120,50


**Observaciones del dataset original**

- Las columnas `6` (minutos por día) y `7` (porcentaje) se leen como texto (`object`/`str`), lo que indica que hay respuestas escritas a mano como `"240 minitos"` o `"Mitad del tiempo"`.
- Los encabezados traen **espacios sobrantes al final**. Esto rompería el renombrado si no se corrige.
- Las columnas `2` y `3` son de texto libre / selección múltiple: se esperan inconsistencias.
- No hay una columna de carrera, así que el diccionario no la incluye.

## 1. Estandarización de columnas

**Decisión:** primero `strip()` a los encabezados y después renombrar con el diccionario a `snake_case` corto.

**Por qué:** el formulario exporta preguntas con espacios finales (`"... frecuencia?  "`). Si renombrás sin limpiarlos, la clave del diccionario no coincide y la columna queda sin renombrar sin dar error. Además se avisa si queda alguna columna sin mapear.

In [13]:
df = df_raw.copy()
df.columns = df.columns.str.strip()

sin_mapeo = set(df.columns) - set(RENOMBRAR)
if sin_mapeo:
    print("[AVISO] Columnas sin mapeo (se conservan):", sin_mapeo)

df = df.rename(columns=RENOMBRAR)
df.columns.tolist()

['marca_temporal',
 'edad',
 'herramienta_ia',
 'finalidad',
 'nivel_experiencia',
 'frecuencia_semanal',
 'tiempo_uso',
 'porcentaje_estudio_ia']

## 2. Limpieza de texto y categorías

**Decisiones**

| Qué | Cómo | Por qué |
|---|---|---|
| Formato base | `lower()` + `strip()` + colapsar espacios internos | `"ChatGPT "`, `"chatgpt"` y `"Chat GPT"` deben contar como **una** categoría |
| Celdas solo con espacios | Se convierten en nulo | Una celda `" "` no es una respuesta; se tratará en el paso de nulos |
| `herramienta_ia` (texto libre) | Se detecta la herramienta por palabra clave | Ej.: `"Cuando se acaba la respuesta, uso gemini"` → `gemini`. Si no coincide ningún patrón, se conserva el texto original en vez de inventar |
| `finalidad` (selección múltiple) | Se separa, se quita el punto final, se ordena y se une con `" \| "` | El formulario guarda `"estudio."` y `"estudio"` según la posición de la opción; ordenar hace que la misma combinación siempre tenga el mismo texto |
| `nivel_experiencia` | Se conserva solo la primera palabra (`básico`/`intermedio`/`avanzado`) | La descripción entre paréntesis es parte de la opción del formulario, no un valor distinto |

**Por qué antes de nulos y duplicados:** hay que normalizar el texto primero para que dos respuestas que solo difieren en mayúsculas o espacios se detecten como iguales.

> **Sobre las minúsculas:** acá se usan solo como **forma canónica interna** para agrupar bien. La versión para mostrar en el dashboard se arma en el paso 7.

In [14]:
def limpiar_texto(df, columnas):
    """lower + strip + espacios internos colapsados. Vacíos -> NA."""
    df = df.copy()
    for col in columnas:
        df[col] = (
            df[col].astype("string")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace("", pd.NA)
        )
    return df


PATRONES_HERRAMIENTA = {
    "chatgpt": r"chat\s*-?\s*gpt|gpt",
    "gemini": r"gemini",
    "claude": r"claude",
    "deepseek": r"deep\s*seek",
    "copilot": r"copilot",
}

def normalizar_herramienta(valor):
    if pd.isna(valor):
        return valor
    for nombre, patron in PATRONES_HERRAMIENTA.items():
        if re.search(patron, valor):
            return nombre
    return valor  # si no se reconoce, se conserva


def normalizar_finalidad(valor):
    if pd.isna(valor):
        return valor
    # separa en ".," o en una coma seguida de letra minúscula
    partes = re.split(r"\.\s*,\s*|,\s*(?=[a-záéíóúñ])", valor)
    partes = sorted({p.strip().rstrip(".").strip() for p in partes if p.strip()})
    return " | ".join(partes)


df = limpiar_texto(df, COLS_CATEGORICAS)
df["herramienta_ia"] = df["herramienta_ia"].map(normalizar_herramienta)
df["finalidad"] = df["finalidad"].map(normalizar_finalidad)
df["nivel_experiencia"] = df["nivel_experiencia"].str.extract(r"^(\w+)")[0]

print(df["herramienta_ia"].value_counts(dropna=False), "\n")
print(df["nivel_experiencia"].value_counts(dropna=False))

herramienta_ia
gemini      21
chatgpt      8
claude       6
deepseek     1
alexa        1
Name: count, dtype: Int64 

nivel_experiencia
básico        20
intermedio    12
avanzado       5
Name: count, dtype: Int64


In [15]:
# Revisión de las combinaciones de finalidad resultantes
df["finalidad"].value_counts(dropna=False)

finalidad
resolver dudas o conceptos de estudio                                                                                                                                    8
escribir o corregir código (programación) | resolver dudas o conceptos de estudio                                                                                        7
búsqueda de información general | escribir o corregir código (programación) | resolver dudas o conceptos de estudio                                                      5
búsqueda de información general | escribir o corregir código (programación) | redacción y resumen de textos | resolver dudas o conceptos de estudio                      5
búsqueda de información general                                                                                                                                          3
entretenimiento                                                                                                                        

## 3. Conversión de tipos (casteo)

**Decisiones**

- `a_numerico()` **extrae el primer número** del texto: `"240 minitos"` → `240`, `"10min"` → `10`.
- Frases sin dígitos se traducen con un diccionario explícito: `"Mitad del tiempo"` → `50`, `"Todo el tiempo si es posible."` → `100`.
- Lo que no se puede interpretar (`"No sé"`) pasa a `NaN`, **no** a un valor inventado. Se decide qué hacer con él en el paso 4.
- `marca_temporal` se convierte a fecha con `dayfirst=True`, porque Google Forms exporta `d/m/aaaa`.

**Por qué antes de los nulos:** el casteo es lo que revela los nulos "ocultos". Una respuesta como `"No sé"` no figura como vacía en el CSV, pero no es un número.

**Por qué no `pd.to_numeric` directo:** fallaría o devolvería `NaN` en respuestas que sí tienen información (`"240 minitos"`), y se perderían datos válidos.

In [16]:
TEXTO_A_PORCENTAJE = {"mitad del tiempo": 50, "todo el tiempo": 100}

def a_numerico(serie, equivalencias=None):
    equivalencias = equivalencias or {}

    def convertir(valor):
        if pd.isna(valor):
            return np.nan
        texto = str(valor).strip().lower()
        for clave, num in equivalencias.items():
            if clave in texto:
                return num
        m = re.search(r"\d+(?:[.,]\d+)?", texto)
        return float(m.group().replace(",", ".")) if m else np.nan

    return pd.to_numeric(serie.map(convertir), errors="coerce")


# Guardamos el original para poder auditar qué se transformó
originales = df[["tiempo_uso", "porcentaje_estudio_ia"]].copy()

df["edad"] = a_numerico(df["edad"])
df["frecuencia_semanal"] = a_numerico(df["frecuencia_semanal"])
df["tiempo_uso"] = a_numerico(df["tiempo_uso"])
df["porcentaje_estudio_ia"] = a_numerico(df["porcentaje_estudio_ia"], TEXTO_A_PORCENTAJE)
df["marca_temporal"] = pd.to_datetime(df["marca_temporal"], dayfirst=True, errors="coerce")

df.dtypes

marca_temporal           datetime64[us]
edad                            float64
herramienta_ia                   string
finalidad                        string
nivel_experiencia                string
frecuencia_semanal              float64
tiempo_uso                      float64
porcentaje_estudio_ia           float64
dtype: object

In [17]:
# Auditoría: respuestas que NO eran un número "limpio" y cómo quedaron
def es_texto(s):
    return ~s.astype(str).str.strip().str.fullmatch(r"\d+(\.\d+)?")

for col in ["tiempo_uso", "porcentaje_estudio_ia"]:
    cambio = es_texto(originales[col])
    print(f"--- {col} ---")
    print(pd.DataFrame({"original": originales.loc[cambio, col], "convertido": df.loc[cambio, col]}), "\n")

--- tiempo_uso ---
       original  convertido
10  240 minitos       240.0
11       10min         10.0
22       180min       180.0
23    30minutos        30.0
34  360 minitos       360.0 

--- porcentaje_estudio_ia ---
                          original  convertido
5                Mitad del tiempo         50.0
23  Todo el tiempo si es posible.        100.0
26                          No sé          NaN 



## 4. Datos faltantes

**Estrategia por tipo de variable**

| Tipo | Acción | Justificación |
|---|---|---|
| Crítica (`herramienta_ia`, `tiempo_uso`, `frecuencia_semanal`) | **Eliminar la fila** | Son las variables centrales del estudio; imputarlas distorsionaría directamente los resultados |
| Numérica no crítica | **Mediana** | Es robusta a valores extremos (hay respuestas de 240 y 360 min), a diferencia de la media |
| Categórica | **`"no responde"`** | No se inventa una categoría; se deja visible para el análisis |

Cada imputación numérica deja una columna `*_imputado` (booleana) para poder excluir esas filas o hacer análisis de sensibilidad.

**Alternativa considerada:** dejar el `NaN` sin imputar. Se descartó porque muchas librerías de gráficos descartan la fila en silencio, pero se puede revertir filtrando por la columna `*_imputado`.

In [18]:
print("Nulos por columna:")
nulos = df.isna().sum()
display(nulos.to_frame("nulos").assign(porcentaje=(nulos / len(df) * 100).round(1)))

Nulos por columna:


,nulos,porcentaje
marca_temporal,0,0.0
edad,0,0.0
herramienta_ia,0,0.0
finalidad,0,0.0
nivel_experiencia,0,0.0
frecuencia_semanal,0,0.0
tiempo_uso,0,0.0
porcentaje_estudio_ia,1,2.7


In [19]:
# 1) Críticos: eliminar fila
antes = len(df)
df = df.dropna(subset=COLS_CRITICAS)
print(f"Filas eliminadas por datos críticos faltantes: {antes - len(df)}")

# 2) Numéricas no críticas: mediana + bandera
for col in COLS_NUMERICAS:
    n = df[col].isna().sum()
    if n:
        mediana = df[col].median()
        df[f"{col}_imputado"] = df[col].isna()
        df[col] = df[col].fillna(mediana)
        print(f"{col}: {n} nulo(s) imputado(s) con mediana = {mediana}")

# 3) Categóricas: "no responde"
for col in COLS_CATEGORICAS:
    n = df[col].isna().sum()
    if n:
        df[col] = df[col].fillna("no responde")
        print(f"{col}: {n} nulo(s) -> 'no responde'")

print("\nNulos restantes:", int(df.isna().sum().sum()))

Filas eliminadas por datos críticos faltantes: 0
porcentaje_estudio_ia: 1 nulo(s) imputado(s) con mediana = 50.0

Nulos restantes: 0


**Observación para revisar:** en `porcentaje_estudio_ia`, la gran mayoría respondió exactamente `50`. Puede ser real, o un efecto de cómo se presentó la pregunta (valor por defecto, opciones). Conviene tenerlo presente al interpretar esa variable.

## 5. Datos duplicados

**Decisiones**

1. Se eliminan las filas **100% idénticas**, conservando la primera (`keep="first"`), como se pidió.
2. Se **avisa sin borrar** si hay respuestas iguales en todo salvo la `marca_temporal`.

**Por qué dos controles:** cada envío del formulario tiene su propia fecha y hora, así que casi nunca habrá filas idénticas *incluyendo* la marca temporal. Pero dos personas distintas pueden responder exactamente igual (sobre todo con pocas preguntas y opciones cerradas). Borrarlas automáticamente podría eliminar respuestas legítimas; por eso la segunda revisión queda para decisión manual.

In [20]:
n_exactos = df.duplicated(keep="first").sum()
print(f"Duplicados exactos: {n_exactos}")
df = df.drop_duplicates(keep="first")

cols_sin_fecha = [c for c in df.columns if c != "marca_temporal"]
mascara = df.duplicated(subset=cols_sin_fecha, keep=False)
print(f"Filas con respuestas idénticas (distinta marca temporal): {mascara.sum()}")
df[mascara].sort_values(cols_sin_fecha)

Duplicados exactos: 0
Filas con respuestas idénticas (distinta marca temporal): 2


,marca_temporal,edad,herramienta_ia,finalidad,nivel_experiencia,frecuencia_semanal,tiempo_uso,porcentaje_estudio_ia,porcentaje_estudio_ia_imputado
0,2026-10-09 20:39:47,27.0,claude,escribir o corregir código (programación) | re...,avanzado,5.0,30.0,50.0,False
15,2026-10-16 14:38:22,27.0,claude,escribir o corregir código (programación) | re...,avanzado,5.0,30.0,50.0,False


In [21]:
df = df.reset_index(drop=True)
print("Filas tras eliminar duplicados exactos:", len(df))

Filas tras eliminar duplicados exactos: 37


## 6. Control de rangos

**Decisión:** solo **informar** los valores fuera de un rango razonable, **sin corregirlos**.

**Por qué:** un valor atípico puede ser un error de tipeo (¿4 minutos o 4 horas?) o una respuesta real. Corregirlo sin poder consultar a la persona sería inventar datos. Se deja la decisión al investigador.

In [22]:
REGLAS = {
    "edad": (15, 80),
    "frecuencia_semanal": (0, 7),
    "tiempo_uso": (5, 720),          # minutos por día
    "porcentaje_estudio_ia": (0, 100),
}

hubo = False
for col, (lo, hi) in REGLAS.items():
    fuera = df[(df[col] < lo) | (df[col] > hi)]
    if len(fuera):
        hubo = True
        print(f"'{col}' fuera de [{lo}, {hi}]:")
        display(fuera[["marca_temporal", col]])
if not hubo:
    print("Ningún valor fuera de rango.")

'tiempo_uso' fuera de [5, 720]:


,marca_temporal,tiempo_uso
26,2026-10-24 10:57:19,4.0


## 7. Formato de presentación para el dashboard

**¿Conviene dejar todo en minúsculas? No para lo que se muestra.**

Las minúsculas sirven para **limpiar y agrupar**, pero en un dashboard quedan mal: `chatgpt`, `deepseek` o `básico` en un gráfico se ven descuidados, y marcas como *ChatGPT* o *DeepSeek* tienen una grafía propia.

**Decisión:** se usa un proceso en dos etapas.

1. **Canónico (minúsculas):** pasos 2 a 6, para agrupar sin errores.
2. **Presentación (este paso):** se convierte a etiquetas legibles con diccionarios explícitos.

**Por qué diccionarios y no `.title()`:** `.title()` daría `Chatgpt` y `Deepseek`, y pondría en mayúscula cada palabra de las finalidades. El diccionario permite controlar la grafía exacta. Cualquier valor que no esté en el diccionario recibe `.capitalize()` como respaldo, así no se pierde.

**Finalidad en columnas booleanas:** como es de selección múltiple, una columna de texto con varias opciones es difícil de graficar (no se puede contar "cuántos usan IA para programar" sin separar el texto). Se agregan columnas `True/False` por finalidad, que en un dashboard se resumen con una simple suma.

In [23]:
ETIQUETA_HERRAMIENTA = {
    "chatgpt": "ChatGPT",
    "gemini": "Gemini",
    "claude": "Claude",
    "deepseek": "DeepSeek",
    "copilot": "Copilot",
    "alexa": "Alexa",
    "no responde": "No responde",
}

ETIQUETA_NIVEL = {
    "básico": "Básico",
    "intermedio": "Intermedio",
    "avanzado": "Avanzado",
    "no responde": "No responde",
}
ORDEN_NIVEL = ["Básico", "Intermedio", "Avanzado"]  # útil para ordenar ejes en el dashboard

# clave canónica -> (nombre de columna booleana, etiqueta para mostrar)
FINALIDADES = {
    "resolver dudas o conceptos de estudio": ("finalidad_estudio", "Resolver dudas o conceptos de estudio"),
    "escribir o corregir código (programación)": ("finalidad_codigo", "Escribir o corregir código"),
    "búsqueda de información general": ("finalidad_busqueda", "Búsqueda de información general"),
    "redacción y resumen de textos": ("finalidad_redaccion", "Redacción y resumen de textos"),
    "entretenimiento": ("finalidad_entretenimiento", "Entretenimiento"),
}

df_final = df.copy()

# a) Columnas booleanas por finalidad (se calculan sobre el texto canónico)
for clave, (col, _) in FINALIDADES.items():
    df_final[col] = df_final["finalidad"].str.contains(clave, regex=False)

# b) Etiquetas legibles (con respaldo .capitalize() para valores no previstos)
df_final["herramienta_ia"] = df_final["herramienta_ia"].map(
    lambda v: ETIQUETA_HERRAMIENTA.get(v, str(v).capitalize())
)
df_final["nivel_experiencia"] = df_final["nivel_experiencia"].map(
    lambda v: ETIQUETA_NIVEL.get(v, str(v).capitalize())
)

def etiquetar_finalidad(valor):
    partes = [FINALIDADES.get(p, (None, p.capitalize()))[1] for p in valor.split(" | ")]
    return " | ".join(partes)

df_final["finalidad"] = df_final["finalidad"].map(etiquetar_finalidad)

# c) Aviso si alguna finalidad no coincidió con ninguna columna booleana
cols_fin = [c for c, _ in FINALIDADES.values()]
sin_match = ~df_final[cols_fin].any(axis=1)
print("Filas sin ninguna finalidad reconocida:", int(sin_match.sum()))

df_final.head()

Filas sin ninguna finalidad reconocida: 0


,marca_temporal,edad,herramienta_ia,finalidad,nivel_experiencia,frecuencia_semanal,tiempo_uso,porcentaje_estudio_ia,porcentaje_estudio_ia_imputado,finalidad_estudio,finalidad_codigo,finalidad_busqueda,finalidad_redaccion,finalidad_entretenimiento
0,2026-10-09 20:39:47,27.0,Claude,Escribir o corregir código | Resolver dudas o ...,Avanzado,5.0,30.0,50.0,False,True,True,False,False,False
1,2026-10-09 20:40:15,20.0,Gemini,Entretenimiento,Básico,5.0,30.0,45.0,False,False,False,False,False,True
2,2026-10-09 20:45:11,20.0,Gemini,Resolver dudas o conceptos de estudio,Básico,2.0,20.0,50.0,False,True,False,False,False,False
3,2026-10-09 20:57:06,33.0,Gemini,Búsqueda de información general | Escribir o c...,Intermedio,5.0,60.0,50.0,False,True,True,True,False,False
4,2026-10-09 21:01:08,28.0,ChatGPT,Redacción y resumen de textos | Resolver dudas...,Intermedio,3.0,120.0,50.0,False,True,False,False,True,False


In [24]:
# Verificación rápida de lo que verá el dashboard
print(df_final["herramienta_ia"].value_counts(), "\n")
print(df_final["nivel_experiencia"].value_counts().reindex(ORDEN_NIVEL), "\n")
print(df_final[cols_fin].sum().rename("personas por finalidad"))

herramienta_ia
Gemini      21
ChatGPT      8
Claude       6
DeepSeek     1
Alexa        1
Name: count, dtype: Int64 

nivel_experiencia
Básico        20
Intermedio    12
Avanzado       5
Name: count, dtype: Int64 

finalidad_estudio            31
finalidad_codigo             20
finalidad_busqueda           17
finalidad_redaccion           9
finalidad_entretenimiento     4
Name: personas por finalidad, dtype: Int64


## 8. Exportación

**Decisión:** `encoding="utf-8"` e `index=False`.

**Por qué:** UTF-8 conserva tildes y la ñ. Si al abrir el CSV en Excel se ven mal, importalo con codificación UTF-8, o cambiá a `encoding="utf-8-sig"`. `index=False` evita una columna `Unnamed: 0` sobrante al volver a leerlo.

In [25]:
df_final.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print(f"Exportado: {RUTA_SALIDA} ({df_final.shape[0]} filas x {df_final.shape[1]} columnas)")
df_final.dtypes

Exportado: datos_limpios_ia.csv (37 filas x 14 columnas)


marca_temporal                    datetime64[us]
edad                                     float64
herramienta_ia                            string
finalidad                                 string
nivel_experiencia                         string
frecuencia_semanal                       float64
tiempo_uso                               float64
porcentaje_estudio_ia                    float64
porcentaje_estudio_ia_imputado              bool
finalidad_estudio                        boolean
finalidad_codigo                         boolean
finalidad_busqueda                       boolean
finalidad_redaccion                      boolean
finalidad_entretenimiento                boolean
dtype: object

## Resumen de decisiones

| # | Paso | Decisión | Riesgo / alternativa |
|---|---|---|---|
| 1 | Columnas | `strip` de encabezados + diccionario `snake_case` | Si el formulario cambia una pregunta, avisa de columnas sin mapear |
| 2 | Texto | Minúsculas + `strip` como forma canónica; palabras clave para texto libre | Una herramienta nueva no reconocida se conserva tal cual |
| 3 | Casteo | Extraer número del texto; frases traducidas con diccionario | `"No sé"` → `NaN` (no se inventa) |
| 4 | Nulos | Críticos: eliminar fila. Numéricas: mediana + bandera. Categóricas: `"no responde"` | Se puede filtrar por `*_imputado` |
| 5 | Duplicados | Eliminar exactos; avisar (sin borrar) los que solo difieren en la fecha | Revisión manual |
| 6 | Rangos | Solo informar, no corregir | Decide el investigador |
| 7 | Presentación | Etiquetas legibles con diccionario + columnas booleanas de finalidad | Minúsculas solo para limpiar |
| 8 | Exportación | UTF-8, sin índice | `utf-8-sig` si Excel muestra mal las tildes |